In [ ]:

import pandas as pd
df=pd.read_csv("../OPENACTIVE_MERGED.csv")
gap_raw=pd.read_csv("../ActiveLives_Data/gapscore_v2.csv")
supply=df.groupby("borough")["session_count"].sum(min_count=1).reset_index(name="sessions")
venues=df.groupby("borough")["location_name"].nunique().reset_index(name="venues")
supply=supply.merge(venues,on="borough",how="outer")
print(supply.shape)

In [ ]:
# demand combiness readiness and inactivity
demand=(gap_raw[gap_raw["covid_affected"] == False].groupby("borough").agg(inactive=("pct_inactive","mean"),readiness_opportunity=("readiness_opportunity_wtd","mean"),readiness_ability=("readiness_ability_wtd","mean"),respondents=("respondents","sum")).reset_index())
print(demand.shape)
print(demand.head())

In [ ]:
#merge demand and supply
g=demand.merge(supply,on="borough",how="outer")
g["sessions"]=g["sessions"].fillna(0)
g["venues"]=g["venues"].fillna(0)
full=g.copy()  
g=g[g["borough"] != "City of London"].copy()
print(g.shape)

In [ ]:
# rescale readiness 
g["readiness_opportunity_pct"]=(g["readiness_opportunity"] - 1) / 4 * 100
g["inactive_rank"]=g["inactive"].rank(pct=True) * 100
g["readiness_opp_rank"]=g["readiness_opportunity_pct"].rank(pct=True) * 100
WEIGHT_INACTIVE=0.5
WEIGHT_READINESS=0.5
g["demand_score"]=(g["inactive_rank"] * WEIGHT_INACTIVE) + (g["readiness_opp_rank"] * WEIGHT_READINESS)
print(g[["borough","inactive","readiness_opportunity_pct","demand_score"]].sort_values("demand_score",ascending=False))

In [ ]:
# Classification 
matrix={
    ("high","low"): "genuine desert",
    ("high","mid"): "emerging desert",
    ("high","high"): "under-monitored",
    ("mid","low"): "blind spot risk",
    ("mid","mid"): "average",
    ("mid","high"): "well-served (moderate need)",
    ("low","low"): "low need, low supply",
    ("low","mid"): "adequately served",
    ("low","high"): "well-served",}

def classify(row,s_col,d_col):
    if row["sessions"] == 0:
        return "blind spot" if row[d_col] in ["mid","high"] else "low need, low supply"
    return matrix.get((row[d_col],row[s_col]),"unclassified")
def bin_col(series,n,labels):
    return pd.qcut(series.rank(method="first"),n,labels=labels)
def bin_col_tiebreak(series,tie_break,n,labels):
    order=pd.DataFrame({"series": series,"tie_break": tie_break}).sort_values(
        ["series","tie_break"],kind="mergesort")
    rank=pd.Series(range(1,len(order) + 1),index=order.index).reindex(series.index)
    return pd.qcut(rank,n,labels=labels)

In [ ]:
# actual classification using demand_score 
g["s_bin"]=bin_col(g["sessions"],3,["low","mid","high"])
g["d_bin"]=bin_col_tiebreak(g["demand_score"],g["inactive"],3,["low","mid","high"])
g["class"]=g.apply(lambda r: classify(r,"s_bin","d_bin"),axis=1)
print(g.sort_values("demand_score",ascending=False)[["borough","demand_score","inactive","readiness_opportunity_pct","sessions","class"]])

In [ ]:
# if values equal use inactivity to rank

def bin_col_tiebreak(series,tie_break,n,labels):
    order=pd.DataFrame({"series": series,"tie_break": tie_break}).sort_values(
        ["series","tie_break"],kind="mergesort")
    rank=pd.Series(range(1,len(order) + 1),index=order.index).reindex(series.index)
    return pd.qcut(rank,n,labels=labels)
g["d_bin_tiebreak"]=bin_col_tiebreak(g["demand_score"],g["inactive"],3,["low","mid","high"])
g["class_tiebreak"]=g.apply(lambda r: classify(r,"s_bin","d_bin_tiebreak"),axis=1)

changed=g[g["class"] != g["class_tiebreak"]].sort_values("demand_score")
print(f"boroughs affected by tie-break fix: {len(changed)} / {len(g)}")
print(changed[["borough","demand_score","inactive","d_bin","d_bin_tiebreak","class","class_tiebreak"]].to_string(index=False))

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min()) * 100
g["inactive_norm"]=minmax(g["inactive"])
g["readiness_opp_norm"]=minmax(g["readiness_opportunity_pct"])

def build_weighted(gdf,inactive_col,readiness_col,w_inactive,w_readiness):
    score=gdf[inactive_col] * w_inactive + gdf[readiness_col] * w_readiness
    d_bin=bin_col_tiebreak(score,gdf["inactive"],3,["low","mid","high"])
    return score,d_bin

def class_with_dbin(gdf,d_bin):
    tmp=gdf.copy()
    tmp["d_bin_tmp"]=d_bin
    return tmp.apply(lambda r: classify(r,"s_bin","d_bin_tmp"),axis=1)

# checking if stable using ranks
rank_score_50,rank_dbin_50=build_weighted(g,"inactive_rank","readiness_opp_rank",0.5,0.5)
rank_score_60,rank_dbin_60=build_weighted(g,"inactive_rank","readiness_opp_rank",0.6,0.4)
rank_score_40,rank_dbin_40=build_weighted(g,"inactive_rank","readiness_opp_rank",0.4,0.6)
rank_class_50=class_with_dbin(g,rank_dbin_50)
rank_class_60=class_with_dbin(g,rank_dbin_60)
rank_class_40=class_with_dbin(g,rank_dbin_40)

#checking stable using min/max
norm_score_50,norm_dbin_50=build_weighted(g,"inactive_norm","readiness_opp_norm",0.5,0.5)
norm_score_60,norm_dbin_60=build_weighted(g,"inactive_norm","readiness_opp_norm",0.6,0.4)
norm_score_40,norm_dbin_40=build_weighted(g,"inactive_norm","readiness_opp_norm",0.4,0.6)
norm_class_50=class_with_dbin(g,norm_dbin_50)
norm_class_60=class_with_dbin(g,norm_dbin_60)
norm_class_40=class_with_dbin(g,norm_dbin_40)
rank_changed_60=(rank_class_50 != rank_class_60).sum()
rank_changed_40=(rank_class_50 != rank_class_40).sum()
rank_changed_either=((rank_class_50 != rank_class_60) | (rank_class_50 != rank_class_40)).sum()
norm_changed_60=(norm_class_50 != norm_class_60).sum()
norm_changed_40=(norm_class_50 != norm_class_40).sum()
norm_changed_either=((norm_class_50 != norm_class_60) | (norm_class_50 != norm_class_40)).sum()
print(f"RANK-based (current):  0.6/0.4 changes {rank_changed_60}/32, 0.4/0.6 changes {rank_changed_40}/32, either: {rank_changed_either}/32")
print(f"MIN-MAX raw (alt):     0.6/0.4 changes {norm_changed_60}/32, 0.4/0.6 changes {norm_changed_40}/32, either: {norm_changed_either}/32")

#finsl check
same_weight_diff=g.loc[rank_class_50 != norm_class_50,["borough"]].copy()
same_weight_diff["class_rank_50"]=rank_class_50[rank_class_50 != norm_class_50]
same_weight_diff["class_norm_50"]=norm_class_50[rank_class_50 != norm_class_50]
print(f"\nboroughs where min-max-50/50 differs from rank-50/50  {len(same_weight_diff)}/32")
print(same_weight_diff.to_string(index=False))

In [ ]:
#check if inactivity and readiness differs
pure_inactive_score,pure_inactive_dbin=build_weighted(g,"inactive_rank","readiness_opp_rank",1.0,0.0)
pure_readiness_score,pure_readiness_dbin=build_weighted(g,"inactive_rank","readiness_opp_rank",0.0,1.0)

tertile_order={"low": 0,"mid": 1,"high": 2}
compare=g[["borough"]].copy()
compare["d_bin_pure_inactive"]=pure_inactive_dbin.astype(str)
compare["d_bin_pure_readiness"]=pure_readiness_dbin.astype(str)
compare["tertile_gap"]=(compare["d_bin_pure_inactive"].map(tertile_order) - compare["d_bin_pure_readiness"].map(tertile_order)).abs()
print(compare.sort_values("tertile_gap",ascending=False).to_string(index=False))
n_full_flip=(compare["tertile_gap"] == 2).sum()
n_any_diff=(compare["d_bin_pure_inactive"] != compare["d_bin_pure_readiness"]).sum()
print(f"\nboroughs that changeentirely {n_full_flip}/32")
print(f"boroughs with any difference  {n_any_diff}/32")
print(f"\ncorrelation betweeninactive and readiness_opportunity_pct {g['inactive'].corr(g['readiness_opportunity_pct']):.3f}")

In [ ]:
#check if sensitive to weights
g["inactive_tertile"]=bin_col_tiebreak(g["inactive_rank"],g["inactive"],3,["low","mid","high"])
g["readiness_tertile"]=bin_col_tiebreak(g["readiness_opp_rank"],g["inactive"],3,["low","mid","high"])

def or_rule(row):
    if row["inactive_tertile"] == "high" or row["readiness_tertile"] == "high":
        return "high"
    elif row["inactive_tertile"] == "low" and row["readiness_tertile"] == "low":
        return "low"
    else:
        return "mid"

g["d_bin_or"]=g.apply(or_rule,axis=1)
g["class_or"]=g.apply(lambda r: classify(r,"s_bin","d_bin_or"),axis=1)

or_vs_current=g[["borough","d_bin_or","class_or","d_bin","class"]].copy()
or_changed=or_vs_current[or_vs_current["class_or"] != or_vs_current["class"]]

print(f"{len(or_changed)}/32 boroughs classified differently")
print(or_changed.to_string(index=False))
print(g["d_bin_or"].value_counts())

In [ ]:
# Sensitivity check 
concern={
"genuine desert": "high","blind spot": "high",
    "emerging desert": "mid","under-monitored": "mid","blind spot risk": "mid","average": "low","well-served (moderate need)": "low","adequately served": "low","well-served": "low","low need, low supply": "low",}

g["s_bin2"]=bin_col(g["sessions"],2,["low","high"])
g["d_bin2"]=bin_col(g["demand_score"],2,["low","high"])
median_matrix={("high","low"): "genuine desert",("high","high"): "under-monitored",("low","low"): "low need, low supply",("low","high"): "well-served"}
g["class_median"]=g.apply(lambda r: "blind spot" if r["sessions"] == 0 and r["d_bin2"] == "high"
                          else median_matrix.get((r["d_bin2"],r["s_bin2"]),"unclassified"),axis=1)

In [ ]:

g["s_bin4"]=bin_col(g["sessions"],4,["low","mid_low","mid_high","high"])
g["d_bin4"]=bin_col(g["demand_score"],4,["low","mid_low","mid_high","high"])
quartile_matrix={("high","low"): "genuine desert",("high","mid_low"): "emerging desert",("high","mid_high"): "under-monitored",("high","high"): "under-monitored",("mid_high","low"): "blind spot risk",("mid_high","mid_low"): "average",("mid_high","mid_high"): "average",("mid_high","high"): "well-served (moderate need)",("mid_low","low"): "blind spot risk",("mid_low","mid_low"): "average",("mid_low","mid_high"): "average",("mid_low","high"): "well-served (moderate need)",("low","low"): "low need, low supply",("low","mid_low"): "adequately served",("low","mid_high"): "adequately served",("low","high"): "well-served",}
g["class_quartile"]=g.apply(lambda r: "blind spot" if r["sessions"] == 0 and r["d_bin4"] in ["mid_high","high"]
    else quartile_matrix.get((r["d_bin4"],r["s_bin4"]),"unclassified"),axis=1)

In [ ]:
# unstable boroughs if changes under diffrent threshold
g["concern"]=g["class"].map(concern)
g["concern_median"]=g["class_median"].map(concern)
g["concern_quartile"]=g["class_quartile"].map(concern)
g["stable"]=(g["concern"] == g["concern_median"]) & (g["concern"] == g["concern_quartile"])
print("unstable boroughs (sensitive to threshold choice):")
print(g.loc[~g["stable"],["borough","concern","concern_median","concern_quartile"]])

In [ ]:
# check bayesian shrinkage 
london_avg=(g["demand_score"] * g["respondents"]).sum() / g["respondents"].sum()
k=g["respondents"].median()
g["demand_score_shrunk"]=((g["demand_score"] * g["respondents"]) + (london_avg * k)) / (g["respondents"] + k)
g["d_bin_shrunk"]=bin_col(g["demand_score_shrunk"],3,["low","mid","high"])
changed=g[g["d_bin"] != g["d_bin_shrunk"]]
print("boroughs that moved tertile after shrinkage:")
print(changed[["borough","demand_score","demand_score_shrunk"]])

In [ ]:
# over riding manually
g.loc[g["borough"]== "Enfield","class"]="blind spot"
g.loc[g["borough"]== "Enfield","note"]="Manually classified as blind spot as facilities exist but are underreported due to a known issue with the Better/GLL data feed."
g.loc[g["borough"]== "Redbridge","note"]="Classified as a blind spot. real facilities exist nearby, but aren't listed on any online booking platform we track."
g.loc[g["borough"]== "Barking and Dagenham","note"]="supply looks strong, but survey data still shows high inactivity, suggesting a non-supply barrier to access."

g.to_csv("../GAP_data/gap_scores.csv",index=False)
full.to_csv("../GAP_data/gap_scores_full33.csv",index=False)
print("saved GAP_data/gap_scores.csv and GAP_data/gap_scores_full33.csv")

In [ ]:
# check correlation 
print(g[['inactive','readiness_opportunity_pct']].corr())
print(g[['demand_score','inactive_rank']].corr())

In [ ]:
df=pd.read_csv("../OPENACTIVE_MERGED.csv")
provider_check=df.groupby('borough')['provider_group'].agg(['count','nunique',lambda x: x.value_counts().to_dict()])
print(provider_check)

In [ ]:
#harringay outlier price
haringey_prices=df[(df['borough']=='Haringey') & (df['price_status']=='known')]
print(haringey_prices[['location_name','price_gbp','activity_type']].sort_values('price_gbp',ascending=False))

In [ ]:
print(df.groupby('activity_type')['price_gbp'].agg(['mean','median','count']))

In [ ]:
other_unspec=df[df['activity_type'] == 'Other / Unspecified']
print(other_unspec.groupby('borough')['price_gbp'].agg(['mean','median','count']).sort_values('mean',ascending=False))